In [ ]:
import pandas as pd
import numpy as np
from src_SVM import *

from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline

# 1.0 - Classificação de Presença Baseada em Fatores Socioeconômicos Usando SVM

In [2]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_SEXO','TP_ESTADO_CIVIL', 'TP_COR_RACA', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'NU_ANO', 'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC']

df = pd.read_parquet("enem_parquet", columns = colunas)

## 1.2- Pré-Processando os Dados

In [3]:
df = pre_processor_SVM(df, objetivo = '', n_samples = 50000)

## 1.3- Construção da Matriz X e Vetor y

In [ ]:
X = df.drop([ 'FALTOU'], axis=1)
y = df['FALTOU']
y = y.map({0: -1, 1: 1})

## 1.4 - Separação em Dados de Treino, Validação e Teste

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

## 1.5 - Tratando os Dados

In [6]:
preprocessador = transformar(X_train)

X_train = preprocessador.transform(X_train).astype(np.float32)
X_val   = preprocessador.transform(X_val).astype(np.float32)
X_test  = preprocessador.transform(X_test).astype(np.float32)

## 1.6 - Treinando o SVM

In [7]:
pipe = Pipeline([
    ('svm', SVC(kernel='rbf', class_weight='balanced', random_state=42))
])

params = {
    'svm__C': [0.001, 0.01, 0.1, 1, 10],
    'svm__gamma': [0.001, 0.01, 0.1, 1]
}

grid = GridSearchCV(pipe, params, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)  

print(grid.best_params_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
{'svm__C': 10, 'svm__gamma': 0.001}


## 1.7 - Validando o SVM

In [9]:
print(classification_report(y_val, grid.predict(X_val)))

              precision    recall  f1-score   support

           0       0.85      0.53      0.65      1748
           1       0.36      0.73      0.49       636

    accuracy                           0.59      2384
   macro avg       0.60      0.63      0.57      2384
weighted avg       0.72      0.59      0.61      2384



## 1.8 Treinando o SVM com Todos os Dados

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = preprocessador.fit_transform(X_train).astype(np.float32)
X_test  = preprocessador.transform(X_test).astype(np.float32) 

svm = SVC(kernel='rbf', class_weight='balanced', random_state=42, C=10, gamma=0.001)

svm.fit(X_train, y_train)

print(classification_report(y_test, svm.predict(X_test)))

              precision    recall  f1-score   support

           0       0.85      0.52      0.64      2177
           1       0.36      0.75      0.49       803

    accuracy                           0.58      2980
   macro avg       0.61      0.63      0.57      2980
weighted avg       0.72      0.58      0.60      2980

